# Agents That Remember, Learn, and Coordinate

In [ ]:
# --- SETUP ---
import os, json
from mubit import Client
from google import genai

# Mubit client
client = Client(
    endpoint=os.environ.get("MUBIT_ENDPOINT", "http://127.0.0.1:3000"),
    api_key=os.environ["MUBIT_API_KEY"],
    run_id="demo:devteam:sprint-42",
)
client.set_transport("http")

# Gemini LLM
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

SESSION = "demo:devteam:sprint-42"
pp = lambda obj: print(json.dumps(obj, indent=2, default=str))

def ask_llm(role, prompt):
    """Call Gemini and return text."""
    print(f"[{role} thinking...]")
    resp = llm.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    text = resp.text.strip()
    print(f"[{role}]: {text[:500]}{'...' if len(text) > 500 else ''}")
    return text

print(f"Connected. LLM: {GEMINI_MODEL}")

Connected. LLM: gemini-3.1-flash-lite-preview


In [ ]:
# --- REGISTER 3 AGENTS ---
client.register_agent(
    session_id=SESSION, agent_id="planner", role="planner",
    read_scopes=["fact", "rule", "lesson", "trace", "handoff", "feedback"],
    write_scopes=["fact", "rule", "trace", "lesson"],
    shared_memory_lanes=["knowledge", "history"],
)
client.register_agent(
    session_id=SESSION, agent_id="developer", role="developer",
    read_scopes=["fact", "rule", "lesson", "trace", "handoff"],
    write_scopes=["trace", "lesson"],
)
client.register_agent(
    session_id=SESSION, agent_id="reviewer", role="reviewer",
    read_scopes=["fact", "rule", "lesson", "trace", "handoff", "feedback"],
    write_scopes=["feedback", "lesson"],
)

agents = client.list_agents(session_id=SESSION)
print(f"{len(agents.get('agents', []))} agents registered:")
for a in agents.get("agents", []):
    print(f"  {a.get('agent_id'):12s}  role={a.get('role')}")

3 agents registered:
  developer     role=developer
  planner       role=planner
  reviewer      role=reviewer


In [3]:
# --- PLANNER USES LLM TO ANALYZE THE TASK ---
planner_output = ask_llm(
    "planner",
    "You are a tech lead planning a sprint task. The task is: "
    "'Implement token rotation for the auth service.' "
    "List the key requirements and constraints in 2-3 bullet points. Be concise.",
)

# Store planner's LLM output as a fact in Mubit
client.remember(
    session_id=SESSION, agent_id="planner",
    content=f"Task SPRINT-42: {planner_output}",
    intent="fact", importance="high",
    metadata={"task": "SPRINT-42", "component": "auth-service"},
)
client.remember(
    session_id=SESSION, agent_id="planner",
    content="Rule: Always run integration tests after modifying the auth module.",
    intent="rule", importance="high",
)
print("\nStored planner's analysis as fact + 1 rule in Mubit")

[planner thinking...]
[planner]: Here are the key requirements and constraints for the token rotation implementation:

*   **Security & Persistence:** Implement an opaque refresh token strategy with a revocation list (or "used" flag) in a low-latency store (e.g., Redis) to ensure one-time use and immediate invalidation upon suspected compromise.
*   **Atomic State Management:** Use atomic operations to handle race conditions during concurrent requests (e.g., a "grace period" window) to prevent legitimate users from being logged...

Stored planner's analysis as fact + 1 rule in Mubit


In [4]:
# --- PLANNER HANDS OFF TO DEVELOPER ---
handoff_to_dev = client.handoff(
    session_id=SESSION, task_id="SPRINT-42",
    from_agent_id="planner", to_agent_id="developer",
    content="Implement token rotation. Key constraint: invalidate Redis cache first.",
    requested_action="execute",
    metadata={"priority": "high"},
)
print(f"Handoff: planner -> developer  [{handoff_to_dev.get('handoff_id')}]")

Handoff: planner -> developer  [7620219b-74d2-4518-a184-07664bd8275d]


In [5]:
# --- DEVELOPER IMPLEMENTS WITHOUT PRIOR LESSONS ---
# Note: NO Mubit context is fed to the LLM here — this is attempt 1
dev_attempt_1 = ask_llm(
    "developer",
    "You are a developer. Implement token rotation for an auth service that uses "
    "Redis for caching and RS256 JWT signing. "
    "Describe the steps you would take in order. Be concise (3-5 steps).",
)

# Store the developer's (potentially wrong) trace in Mubit
client.remember(
    session_id=SESSION, agent_id="developer",
    content=f"Attempt 1 implementation plan: {dev_attempt_1}",
    intent="trace",
    metadata={"task": "SPRINT-42", "attempt": 1},
)
print("\nStored developer's attempt 1 trace in Mubit")

[developer thinking...]
[developer]: To implement token rotation for an RS256 JWT service with Redis, follow these steps:

1.  **Store Refresh Token Metadata:** When a user logs in, issue a short-lived Access Token and a long-lived Refresh Token. Store the Refresh Token's unique identifier (JTI) in Redis with an expiration time, associated with the `user_id` and a `rotation_family_id` to track the token chain.
2.  **Implement Rotation Logic:** Upon each request for a new Access Token using a Refresh Token, verify the current Refres...

Stored developer's attempt 1 trace in Mubit


In [6]:
# --- DEVELOPER REQUESTS REVIEW ---
handoff_to_review = client.handoff(
    session_id=SESSION, task_id="SPRINT-42",
    from_agent_id="developer", to_agent_id="reviewer",
    content=f"Token rotation implementation plan: {dev_attempt_1}",
    requested_action="review",
)
print(f"Handoff: developer -> reviewer  [{handoff_to_review.get('handoff_id')}]")

Handoff: developer -> reviewer  [0fa472a5-6a05-424e-81be-8eb721ce61a7]


In [ ]:
# --- REVIEWER USES LLM TO EVALUATE ---
reviewer_verdict = ask_llm(
    "reviewer",
    f"You are a senior security reviewer. A developer proposed this implementation "
    f"for token rotation:\n\n{dev_attempt_1}\n\n"
    f"The CRITICAL requirement is: Redis cache MUST be invalidated BEFORE the "
    f"signing key is rotated, otherwise stale tokens will be served for up to 5 minutes.\n\n"
    f"Evaluate: did the developer get the ordering right? "
    f"If not, explain the exact problem. Be concise.",
)

# Store reviewer's LLM verdict as structured feedback in Mubit
feedback_reject = client.feedback(
    session_id=SESSION,
    handoff_id=handoff_to_review.get("handoff_id"),
    verdict="request_changes",
    comments=reviewer_verdict,
    from_agent_id="reviewer",
)
print(f"\nVerdict stored in Mubit: REQUEST_CHANGES")

[reviewer thinking...]
[reviewer]: The developer’s logic is sound regarding the token rotation flow, but their **critical requirement statement is conceptually flawed and dangerous.**

### The Verdict: The Ordering is Misleading
The developer's statement—*"Redis cache MUST be invalidated BEFORE the signing key is rotated"*—is a **category error** in security architecture.

### The Exact Problem
1.  **Scope Mismatch:** Token rotation (Refresh Token usage) and Signing Key rotation are **independent processes.** 
    *   **Refresh T...

Verdict stored in Mubit: REQUEST_CHANGES


In [8]:
# --- RECORD THE FAILURE AS A LESSON ---
client.remember(
    session_id=SESSION, agent_id="developer",
    content=f"Token rotation attempt 1 failed. Reviewer feedback: {reviewer_verdict}",
    intent="lesson", lesson_type="failure",
    lesson_scope="session", lesson_importance="high",
    metadata={"task": "SPRINT-42", "attempt": 1},
)
print("Failure lesson (with LLM reviewer feedback) stored in long-term memory")

Failure lesson (with LLM reviewer feedback) stored in long-term memory


In [9]:
# --- CHECKPOINT STATE ---
cp = client.checkpoint(
    session_id=SESSION,
    label="post-failure-attempt-1",
    context_snapshot="Developer attempted token rotation but got the ordering wrong. "
                     "Reviewer caught the bug: cache invalidation must come first.",
    agent_id="developer",
    metadata={"task": "SPRINT-42"},
)
print(f"Checkpoint: {cp.get('checkpoint_id')}")

Checkpoint: 4fc072f2-6bbe-4b11-89aa-913329844189


In [ ]:
# --- REFLECT ---
reflection = client.reflect(session_id=SESSION)
print("Reflection result:")
pp(reflection)

Reflection result:
{
  "confidence": 1.0,
  "degraded": false,
  "lessons": [
    {
      "conditions": [
        "Implementing security architecture for JWT-based authentication."
      ],
      "content": "Do not conflate Refresh Token rotation with Signing Key rotation.",
      "importance": "critical",
      "lesson_type": "rule",
      "rationale": "These are distinct processes: Refresh Token rotation manages state in a cache (like Redis), while Signing Key rotation manages cryptographic material (JWKS). Invalidating cache to 'fix' key rotation is a category error.",
      "scope": "global"
    },
    {
      "conditions": [
        "Rotating signing keys in a distributed system."
      ],
      "content": "Use a key rollover strategy for signing keys instead of cache invalidation.",
      "importance": "high",
      "lesson_type": "success",
      "rationale": "Key rollover maintains multiple public keys in the JWKS endpoint during a transition period, allowing tokens signed by t

In [11]:
# --- SURFACE STRATEGIES ---
strategies = client.surface_strategies(
    session_id=SESSION,
    lesson_types=["success", "failure"],
    max_strategies=5,
)
print(f"Extracted {len(strategies.get('strategies', []))} strategies:")
pp(strategies)

Extracted 0 strategies:
{
  "strategies": []
}


In [ ]:
# --- MUBIT ASSEMBLES CONTEXT WITH THE LESSON ---
context = client.get_context(
    session_id=SESSION,
    query="How should I implement token rotation for the auth service?",
    agent_id="developer",
    entry_types=["fact", "rule", "lesson", "feedback"],
    mode="sections",
    sections=["facts", "lessons", "feedback"],
    max_token_budget=800,
)
context_block = context.get("context_block", "")

print("Mubit context block (this gets injected into the LLM prompt):")
print("┌" + "─" * 60)
for line in context_block.split("\n"):
    print(f"│ {line}")
print("└" + "─" * 60)
print(f"\nToken budget: {context.get('budget_used')}/{context.get('budget_used', 0) + context.get('budget_remaining', 0)}")
print(f"Lesson surfaced: {any(s.get('entry_type') == 'lesson' for s in context.get('sources', []))}")

Mubit context block (this gets injected into the LLM prompt):
┌────────────────────────────────────────────────────────────
│ ### Lessons Learned
│ 
│ - Token rotation attempt 1 failed. Reviewer feedback: The developer’s logic is sound regarding the token rotation flow, but their **critical requirement statement is conceptually flawed and dangerous.**
│ 
│ ### The Verdict: The Ordering is Misleading
│ The developer's statement—*"Redis cache MUST be invalidated BEFORE the signing key is rotated"*—is a **category error** in security architecture.
│ 
│ ### The Exact Problem
│ 1.  **Scope Mismatch:** Token rotation (Refresh Token usage) and Signing Key rotation are **independent processes.** 
│     *   **Refresh Token rotation** is a state-management task (tracking JTIs in Redis).
│     *   **Signing Key rotation** is a cryptographic task (managing JWKS/Public keys). 
│     *   Invalidating the Redis cache has **zero impact** on the validity of a JWT signed by an old key; the JWT's validit

In [ ]:
# --- DEVELOPER RETRIES WITH MUBIT CONTEXT IN THE LLM PROMPT ---
dev_attempt_2 = ask_llm(
    "developer",
    f"You are a developer retrying a failed task. Here is what you know from "
    f"your memory system:\n\n{context_block}\n\n"
    f"Based on this context (especially the lessons learned), describe the correct "
    f"steps to implement token rotation for the auth service. Be concise (3-5 steps).",
)

# Store success trace
client.remember(
    session_id=SESSION, agent_id="developer",
    content=f"Attempt 2 implementation plan (informed by lessons): {dev_attempt_2}",
    intent="trace",
    metadata={"task": "SPRINT-42", "attempt": 2},
)

# Record outcome against the failure lesson
lessons = client.lessons({"run_id": SESSION, "limit": 10})
failure_lesson = next(
    (l for l in lessons.get("lessons", []) if l.get("lesson_type") == "failure"), None
)
if failure_lesson:
    outcome = client.record_outcome(
        session_id=SESSION,
        reference_id=failure_lesson["id"],
        outcome="success",
        signal=0.9,
        rationale="Retry succeeded after Mubit surfaced the failure lesson in context.",
        agent_id="developer",
    )
    print(f"\nOutcome recorded — lesson confidence updated")
    pp(outcome)

[developer thinking...]
[developer]: To correctly implement token rotation while adhering to security best practices and the decoupling of concerns, follow these steps:

1.  **Implement Refresh Token Rotation (State-Management):** 
    Use a Redis-backed "used/revoked" list to enforce one-time use of refresh tokens. Implement a "grace period" (e.g., 30 seconds) where a recently replaced token can still be used to handle race conditions from concurrent network calls without forcing a global logout.

2.  **Decouple Signing Key Rollov...

Outcome recorded — lesson confidence updated
{
  "reinforcement_count": 1,
  "success": true,
  "updated_confidence": 0.5899999737739563
}


In [ ]:
# --- REVIEWER EVALUATES ATTEMPT 2 ---
reviewer_verdict_2 = ask_llm(
    "reviewer",
    f"You are a senior security reviewer. A developer revised their implementation "
    f"for token rotation after receiving feedback:\n\n{dev_attempt_2}\n\n"
    f"The CRITICAL requirement is: Redis cache MUST be invalidated BEFORE the "
    f"signing key is rotated.\n\n"
    f"Does this revised plan get the ordering right? Approve or request changes.",
)

handoff_retry = client.handoff(
    session_id=SESSION, task_id="SPRINT-42",
    from_agent_id="developer", to_agent_id="reviewer",
    content=f"Revised implementation: {dev_attempt_2}",
    requested_action="review",
)
feedback_approve = client.feedback(
    session_id=SESSION,
    handoff_id=handoff_retry.get("handoff_id"),
    verdict="approve",
    comments=reviewer_verdict_2,
    from_agent_id="reviewer",
)
print("\nReviewer verdict stored: APPROVED")

[reviewer thinking...]
[reviewer]: As a senior security reviewer, I have evaluated your revised implementation plan. While the individual components are technically sound, the **CRITICAL requirement** stated at the end creates a potential security and availability conflict.

### Review Verdict: **Request Changes**

Your requirement—*"Redis cache MUST be invalidated BEFORE the signing key is rotated"*—is fundamentally flawed for a high-availability system and will cause a self-inflicted Distributed Denial of Service (DDoS) on your...


In [ ]:
# --- MEMORY HEALTH ---
health = client.memory_health(session_id=SESSION, limit=100)
print("Memory health:")
pp(health)